# Reading Why: train a model

Train the paper's Born Transformer (or the softmax baseline) on e-SNLI and look at its evidence. Use a GPU runtime
(Runtime > Change runtime type > GPU): the paper's 20 epochs take about an hour on a laptop GPU; the default here
is 5 epochs. The data, e-SNLI (about 200 MB), is downloaded from its authors' repository and checked.

In [ ]:
#@title Setup (run this first)
# Get the code (in Colab: a fresh copy of the repository; when opened from a clone: that clone)
import os, sys
if os.path.exists("ui.py"):
    ROOT = ".."
elif os.path.exists("demo/ui.py"):
    ROOT = "."
elif os.path.exists("reading-why/.git"):
    !git -C reading-why pull -q        # a runtime that already has the code: bring it up to date
    ROOT = "reading-why"
else:
    !git clone -q --depth 1 https://github.com/stein-research/reading-why
    ROOT = "reading-why"
sys.path.insert(0, os.path.join(ROOT, "demo"))
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
import importlib, ui, reading_why as rw
importlib.reload(rw); importlib.reload(ui)      # the latest code, also when the runtime already had it
print("ready")

In [ ]:
#@title Get e-SNLI
rw.get_esnli()

In [ ]:
#@title Train
MODEL = "Born-p2-ap"  #@param ["Born-p2-ap", "Softmax", "Poly-sim-born"]
EPOCHS = 5  #@param {type:"integer"}
# 5 epochs: about 20-30 min for the Born Transformer, ~73 % test accuracy. The paper trains 20 (about an hour on a
# laptop GPU; 76.98 +- 0.17 % over three seeds). Choose the number above.
SEED = 0  #@param {type:"integer"}
TAG = f"esnli_{MODEL}_b2_s{SEED}_e{EPOCHS}"
import subprocess
cmd = [sys.executable, "-u", os.path.join("src", "train.py"), "--model", MODEL, "--blocks", "2",   # run in ROOT
       "--epochs", str(EPOCHS), "--lr", "2e-3", "--drop", "0.1", "--seed", str(SEED), "--data", "esnli",
       "--tag", TAG]
with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="")
if p.returncode:
    raise RuntimeError(f"training stopped with code {p.returncode}")

**Check the evidence.** Completeness, deletion (AOPC, flip) and agreement with the annotators on 1,000 test pairs,
as in the paper's Table 1.

In [ ]:
#@title Evaluate
with subprocess.Popen([sys.executable, "-u", os.path.join("src", "faithfulness.py"), TAG], cwd=ROOT,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="")
if p.returncode:
    raise RuntimeError(f"evaluation stopped with code {p.returncode}")

**Explore.** The demo with the model you trained.

In [ ]:
#@title Explore the model you trained
m = rw.load(TAG)                 # your model takes the place of the paper's on its toggle
ui._MODELS["softmax" if m.family == "S" else "born"] = m
print(f"{TAG}: test accuracy {m.test_acc:.4f}")
ui.demo()